In [ ]:
!pip install -q transformers datasets evaluate accelerate peft bitsandbytes sentencepiece sacrebleu

In [ ]:
import torch
import numpy as np
from datasets import load_dataset
from transformers import (
    MT5ForConditionalGeneration,
    AutoTokenizer,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    BitsAndBytesConfig # Added BitsAndBytesConfig for QLoRA
)
import evaluate

# LoRA
from peft import LoraConfig, get_peft_model, TaskType

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
BASE_PATH = "/content/drive/MyDrive/mt5"

data_path = f"{BASE_PATH}/cleaned_data"

TRAIN_PATH = f"{data_path}/train/Final_6M_CM_SelectedColumns_train.jsonl"

OUTPUT_DIR = f"{BASE_PATH}/checkpoints"
LOG_DIR    = f"{BASE_PATH}/logs"

In [ ]:
MODEL_NAME = "google/mt5-small"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Configure 8-bit quantization
quantization_config = BitsAndBytesConfig(load_in_8bit=True)

model = MT5ForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config, # Use quantization_config instead of load_in_8bit
    device_map="auto"
)

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,

    target_modules=["q", "v"],   # important for mT5
    lora_dropout=0.1,

    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

In [ ]:
PREFIX_H2E = "translate Hinglish to English: "
PREFIX_E2H = "translate English to Hinglish: "

In [ ]:
dataset = load_dataset("json", data_files={
    "train": TRAIN_PATH
})

# Select the first 100,000 examples from the training dataset
full_train_dataset = dataset["train"].select(range(100000))

# Split the selected 100,000 examples into training and validation sets
# Adjust test_size as needed for your desired split ratio
split_dataset = full_train_dataset.train_test_split(test_size=0.1, seed=42)

# Assign the splits to the 'dataset' object
dataset["train"] = split_dataset["train"]
dataset["validation"] = split_dataset["test"]

In [ ]:
MAX_INPUT = 128
MAX_TARGET = 128

def preprocess_function(examples):
    inputs = []
    targets = []

    # Corrected: using 'cm_text_r' for Hinglish as per user's clarification
    for eng_text, hin_text in zip(examples["emb_text"], examples["cm_text_r"]):

        # Forward (Hinglish to English - ALWAYS include)
        inputs.append(PREFIX_H2E + hin_text)
        targets.append(eng_text)

        # Backward (English to Hinglish - ONLY sometimes)
        if np.random.rand() < 0.3:   # 30% backward
            inputs.append(PREFIX_E2H + eng_text)
            targets.append(hin_text)

    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT,
        truncation=True,
        padding="max_length"
    )
    labels = tokenizer(
        targets,
        max_length=MAX_TARGET,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [ ]:
# tokenized_datasets = dataset.map(
#     preprocess_function,
#     batched=True,
#     num_proc=2,
#     remove_columns=dataset["train"].column_names
# )

In [ ]:
# # Print the column names of the training set
# print("Column names in the training dataset:", dataset["train"].column_names)

# # Display the first few examples from the training dataset to understand its structure
# print("\nFirst 5 examples from the training dataset:")
# for i in range(min(5, len(dataset["train"]))):
#     print(dataset["train"][i])

**Saving to drive**

In [ ]:
# TOKENIZED_DATA_PATH = f"{BASE_PATH}/tokenized_data_corrected"
# tokenized_datasets.save_to_disk(TOKENIZED_DATA_PATH)
# print(f"Tokenized datasets saved to: {TOKENIZED_DATA_PATH}")

To load the tokenized data in a future session, you can use the following code. This will allow you to skip the tokenization step if the data is already processed and saved.

In [ ]:
from datasets import load_from_disk

TOKENIZED_DATA_PATH = f"{BASE_PATH}/tokenized_data_corrected"
tokenized_datasets = load_from_disk(TOKENIZED_DATA_PATH)
print(f"Tokenized datasets loaded from: {TOKENIZED_DATA_PATH}")

# You might want to remove the 'token_type_ids' column if it's present and not needed for your model.
# Some models don't use this and it can cause issues during training if present in the dataset.
# Example: tokenized_datasets = tokenized_datasets.remove_columns(['token_type_ids'])

# Check the loaded dataset structure
print(tokenized_datasets)

In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

In [ ]:
bleu = evaluate.load("sacrebleu")
chrf = evaluate.load("chrf") # Load chrF metric

def compute_metrics(eval_preds):
    preds, labels = eval_preds

    if isinstance(preds, tuple):
        preds = preds[0]

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # SacreBLEU and chrF expect references as a list of lists of strings
    formatted_decoded_labels = [[l] for l in decoded_labels]

    bleu_result = bleu.compute(predictions=decoded_preds, references=formatted_decoded_labels)
    chrf_result = chrf.compute(predictions=decoded_preds, references=formatted_decoded_labels)

    return {
        "bleu": bleu_result["score"],
        "chrf": chrf_result["score"]
    }

In [ ]:
     # Changed to save to Google Drive

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir=f"{BASE_PATH}/checkpoints_3",

    per_device_train_batch_size=4,   # small due to QLoRA
    per_device_eval_batch_size=4,

    gradient_accumulation_steps=4,   # effective batch = 16

    learning_rate=2e-4,
    num_train_epochs=3,

    # Removed evaluation strategy, steps, and metric_for_best_model
    save_strategy="steps",

    # eval_steps=2000, # Removed
    save_steps=250,

    logging_steps=200,

    predict_with_generate=True,

    fp16=False, # Temporarily set to False to debug zero loss

    save_total_limit=20,

    load_best_model_at_end=False, # Set to False since no evaluation metric is used
    # metric_for_best_model="bleu", # Removed

    report_to="none"
)

In [ ]:
# training_args = Seq2SeqTrainingArguments(
#     output_dir=f"{BASE_PATH}/checkpoints_3",

#     per_device_train_batch_size=4,   # small due to QLoRA
#     per_device_eval_batch_size=4,

#     gradient_accumulation_steps=4,   # 🔥 effective batch = 16

#     learning_rate=2e-4,
#     num_train_epochs=3,

#     eval_strategy="steps", # Enable evaluation at specified steps
#     save_strategy="steps",

#     eval_steps=2000,
#     save_steps=2000, # Changed save_steps to be a multiple of eval_steps

#     logging_steps=200,

#     predict_with_generate=True,

#     fp16=False, # Temporarily set to False to debug zero loss

#     save_total_limit=20,

#     load_best_model_at_end=True, # Load the best model found during training
#     metric_for_best_model="loss", # Track validation loss to find the best model

#     report_to="none"
# )

In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,

    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],

    data_collator=data_collator,

    compute_metrics=compute_metrics
)

In [ ]:
trainer.train(resume_from_checkpoint=True)
# trainer.train(resume_from_checkpoint=False)

# **Evaluation code**

In [ ]:
!pip install -q evaluate sacrebleu unbabel-comet
!pip install -U bitsandbytes>=0.46.1 # Install or upgrade bitsandbytes

In [ ]:
import os
os.kill(os.getpid(), 9)

In [ ]:
import torch
import numpy as np
import os
from tqdm import tqdm

from datasets import load_dataset, load_from_disk
from transformers import (
    MT5ForConditionalGeneration,
    AutoTokenizer,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    BitsAndBytesConfig
)
import evaluate
from peft import LoraConfig, get_peft_model, TaskType, PeftModel

In [ ]:
# =========================================================
# 1. Mount Google Drive
# =========================================================
print("1. Mounting Google Drive...")
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# =========================================================
# 2. Define Paths
# =========================================================
print("2. Defining paths...")
BASE_PATH = "/content/drive/MyDrive/shared/mt5"
data_path = f"{BASE_PATH}/cleaned_data"
TRAIN_PATH = f"{data_path}/train/Final_6M_CM_SelectedColumns_train.jsonl"
OUTPUT_DIR = f"{BASE_PATH}/checkpoints"
TOKENIZED_DATA_PATH = f"{BASE_PATH}/tokenized_data_corrected"
CHECKPOINT_DIR_FOR_EVAL = f"{BASE_PATH}/checkpoints_3"

In [ ]:
# =========================================================
# Define paths for saving evaluation results
# =========================================================
print("Defining paths for saving evaluation outputs...")
EVAL_RESULTS_FILE = f"{BASE_PATH}/evaluation_metrics.json"
PREDICTIONS_FILE = f"{BASE_PATH}/raw_predictions_for_comet.npy"
LABELS_FILE = f"{BASE_PATH}/raw_labels_for_comet.npy"
SOURCES_FILE = f"{BASE_PATH}/sources_for_comet.txt"
COMET_RESULTS_FILE = f"{BASE_PATH}/comet_score.json"


In [ ]:

# Prefixes (ensure these are consistent with training)
PREFIX_H2E = "translate Hinglish to English: "
PREFIX_E2H = "translate English to Hinglish: "
MAX_INPUT = 128
MAX_TARGET = 128

In [ ]:
# =========================================================
# 3. Load Tokenizer
# =========================================================
print("3. Loading tokenizer...")
MODEL_NAME = "google/mt5-small"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
# =========================================================
# 4. Load Base Model with Quantization and LoRA adapters
# =========================================================
print("4. Loading base model with quantization and LoRA adapters...")
quantization_config = BitsAndBytesConfig(load_in_8bit=True)
base_model = MT5ForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto"
)

# Find the latest checkpoint from the training directory
checkpoints = [d for d in os.listdir(CHECKPOINT_DIR_FOR_EVAL) if d.startswith("checkpoint-")]
if not checkpoints:
    raise ValueError(f"No checkpoints found in {CHECKPOINT_DIR_FOR_EVAL}")

# Sort checkpoints by step number to get the latest
latest_checkpoint_name = sorted(checkpoints, key=lambda x: int(x.split('-')[1]))[-1]
latest_checkpoint_path = os.path.join(CHECKPOINT_DIR_FOR_EVAL, latest_checkpoint_name)
print(f"  Loading LoRA adapters from latest checkpoint: {latest_checkpoint_path}")

# Load LoRA adapters onto the base model
model = PeftModel.from_pretrained(base_model, latest_checkpoint_path)
model.eval() # Set model to evaluation mode

In [ ]:
# =========================================================
# 5. Load Datasets (Raw and Tokenized)
# =========================================================
print("5. Loading datasets...")
# Load the raw dataset to reconstruct sources for COMET
raw_dataset_full = load_dataset("json", data_files={"train": TRAIN_PATH})
full_train_dataset_100k = raw_dataset_full["train"].select(range(100000))

# Re-split to match the original dataset structure for 'validation' as our test set
# This ensures the 'validation' split used for evaluation matches the 'dataset' object in prior cells.
# Here, we create a train_val_split, where train_val_split['test'] becomes our 'validation' split.
# We then reference this as the 'testing data' for evaluation.
raw_split_dataset = full_train_dataset_100k.train_test_split(test_size=0.1, seed=42)
raw_dataset = {"train": raw_split_dataset["train"], "validation": raw_split_dataset["test"]}

# Load the pre-tokenized dataset
tokenized_datasets = load_from_disk(TOKENIZED_DATA_PATH)
print(f"  Tokenized datasets loaded from: {TOKENIZED_DATA_PATH}")
print(f"  Using tokenized_datasets['validation'] (num_rows: {len(tokenized_datasets['validation'])}) as testing data.")


In [ ]:

# =========================================================
# 6. Initialize Data Collator
# =========================================================
print("6. Initializing Data Collator...")
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

In [ ]:
# =========================================================
# 7. Load Evaluation Metrics
# =========================================================
print("7. Loading evaluation metrics (sacrebleu, chrf, comet)...")
bleu = evaluate.load("sacrebleu")
chrf = evaluate.load("chrf")
comet = evaluate.load("comet") # Semantic similarity metric

In [ ]:
# =========================================================
# 8. Define compute_metrics function for Trainer
# =========================================================
def compute_metrics(eval_preds):
    preds, labels = eval_preds

    if isinstance(preds, tuple):
        preds = preds[0] # Get predictions from tuple if present

    # Explicitly convert predictions to integer type (long) before decoding.
    # The OverflowError occurs when the tokenizer tries to convert very large float values
    # (potentially from model output or quantization effects) to integers.
    # Rounding and then casting to long should handle most cases.
    try:
        preds = torch.round(torch.tensor(preds)).long()
    except OverflowError as e:
        print(f"Caught OverflowError during prediction conversion: {e}")
        print("This often indicates extremely large values in model predictions.")
        # As a fallback, clamp values to a reasonable range (e.g., 0 to vocab_size-1)
        # This might result in degraded output but prevents a crash for debugging.
        max_token_id = tokenizer.vocab_size - 1
        preds = torch.clamp(torch.round(torch.tensor(preds)), min=0, max=max_token_id).long()


    # Decode predictions
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    # Decode labels (references), handling -100 which are ignored tokens
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # SacreBLEU and chrF expect references as a list of lists of strings
    formatted_decoded_labels = [[l] for l in decoded_labels]

    bleu_result = bleu.compute(predictions=decoded_preds, references=formatted_decoded_labels)
    chrf_result = chrf.compute(predictions=decoded_preds, references=formatted_decoded_labels)

    return {
        "bleu": bleu_result["score"],
        "chrf": chrf_result["score"]
    }

In [ ]:
# =========================================================
# 9. Configure Training Arguments for Evaluation
# =========================================================
print("9. Configuring training arguments for evaluation...")
eval_training_args = Seq2SeqTrainingArguments(
    output_dir="./evaluation_results", # Temporary directory for evaluation outputs
    per_device_eval_batch_size=4,
    predict_with_generate=True,
    fp16=False, # Match training setting
    report_to="none",
    # No saving or logging to drive during evaluation
)

In [ ]:
# =========================================================
# 10. Initialize Trainer for Evaluation
# =========================================================
print("10. Initializing Trainer for evaluation...")
eval_trainer = Seq2SeqTrainer(
    model=model,
    args=eval_training_args,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    eval_dataset=tokenized_datasets["validation"] # Use validation split as testing data
)

In [ ]:
# =========================================================
# 11. Perform Evaluation (Loss, BLEU, chrF)
# =========================================================
print("11. Performing evaluation (Loss, BLEU, chrF)...")
eval_results = eval_trainer.evaluate()

In [ ]:
# =========================================================
# 11.1 Save Evaluation Results
# =========================================================
import json
import os

print(f"Saving evaluation metrics to {EVAL_RESULTS_FILE}...")
with open(EVAL_RESULTS_FILE, "w") as f:
    json.dump(eval_results, f, indent=4)
print("Evaluation metrics saved.")


**resume evaluation**

In [ ]:
# =========================================================
# 12. Generate Predictions for COMET (with resumable saving)
# =========================================================
print("12. Generating predictions for COMET with resumable saving...")

BATCH_SAVE_INTERVAL = 100 # Save every 100 batches
TEMP_PREDS_FILE = f"{BASE_PATH}/temp_predictions.npy"
TEMP_LABELS_FILE = f"{BASE_PATH}/temp_labels.npy"
TEMP_SOURCES_FILE = f"{BASE_PATH}/temp_sources.txt"
LAST_BATCH_INDEX_FILE = f"{BASE_PATH}/last_batch_index.txt"

all_predictions = []
all_labels = []
all_sources = []
start_batch_index = 0

# Check for existing temporary files to resume
if os.path.exists(TEMP_PREDS_FILE) and os.path.exists(TEMP_LABELS_FILE) and os.path.exists(TEMP_SOURCES_FILE) and os.path.exists(LAST_BATCH_INDEX_FILE):
    print("  Resuming prediction from last saved point...")
    all_predictions = np.load(TEMP_PREDS_FILE, allow_pickle=True).tolist()
    all_labels = np.load(TEMP_LABELS_FILE, allow_pickle=True).tolist()
    with open(TEMP_SOURCES_FILE, "r", encoding="utf-8") as f:
        all_sources = [line.strip() for line in f]
    with open(LAST_BATCH_INDEX_FILE, "r") as f:
        start_batch_index = int(f.read()) + 1 # Start from the next batch
    print(f"  Loaded {len(all_predictions)} predictions. Resuming from batch {start_batch_index}.")


# Manual prediction loop
data_loader = eval_trainer.get_eval_dataloader(tokenized_datasets["validation"])

for i, batch in tqdm(enumerate(data_loader), total=len(data_loader), initial=start_batch_index, desc="Generating Predictions"):
    if i < start_batch_index:
        continue # Skip already processed batches if resuming

    # Move batch to device
    with torch.no_grad():
        input_ids = batch["input_ids"].to(model.device)
        attention_mask = batch["attention_mask"].to(model.device)
        labels = batch["labels"].to(model.device)

        # Generate predictions
        outputs = model.generate(
            input_ids=input_ids, # Pass input_ids as a keyword argument
            attention_mask=attention_mask,
            max_new_tokens=MAX_TARGET
        )

        # Collect predictions and labels
        all_predictions.extend(outputs.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

        # Decode source inputs for COMET
        batch_sources = tokenizer.batch_decode(input_ids, skip_special_tokens=True)
        all_sources.extend(batch_sources)

    # Save intermediate progress periodically
    if (i + 1) % BATCH_SAVE_INTERVAL == 0:
        print(f"  Saving intermediate predictions at batch {i + 1}...")
        np.save(TEMP_PREDS_FILE, np.array(all_predictions, dtype=object))
        np.save(TEMP_LABELS_FILE, np.array(all_labels, dtype=object))
        with open(TEMP_SOURCES_FILE, "w", encoding="utf-8") as f:
            for src in all_sources:
                f.write(src + "\n")
        with open(LAST_BATCH_INDEX_FILE, "w") as f:
            f.write(str(i))
        print("  Intermediate predictions saved.")


predictions_token_ids = np.array(all_predictions, dtype=object)
label_token_ids = np.array(all_labels, dtype=object)
sources_for_comet = all_sources

# Remove temporary files after successful completion
if os.path.exists(TEMP_PREDS_FILE): os.remove(TEMP_PREDS_FILE)
if os.path.exists(TEMP_LABELS_FILE): os.remove(TEMP_LABELS_FILE)
if os.path.exists(TEMP_SOURCES_FILE): os.remove(TEMP_SOURCES_FILE)
if os.path.exists(LAST_BATCH_INDEX_FILE): os.remove(LAST_BATCH_INDEX_FILE)

# Decode for final COMET calculation
decoded_preds = tokenizer.batch_decode(predictions_token_ids, skip_special_tokens=True)
decoded_labels = tokenizer.batch_decode(
    np.where(label_token_ids != -100, label_token_ids, tokenizer.pad_token_id),
    skip_special_tokens=True
)

print("  Prediction generation complete.")

In [ ]:
# =========================================================
# 12.1 Save Raw Predictions and Labels
# =========================================================
print(f"Saving raw predictions and labels to {PREDICTIONS_FILE} and {LABELS_FILE}...")
# Ensure predictions_token_ids is a numpy array for saving
predictions_token_ids_np = np.array(predictions_token_ids)
label_token_ids_np = np.array(label_token_ids)

np.save(PREDICTIONS_FILE, predictions_token_ids_np)
np.save(LABELS_FILE, label_token_ids_np)
print("Raw predictions and labels saved.")

# =========================================================
# 12.2 Save Sources for COMET
# =========================================================
print(f"Saving sources for COMET to {SOURCES_FILE}...")
with open(SOURCES_FILE, "w", encoding="utf-8") as f:
    for source in sources_for_comet:
        f.write(source + "\n")
print("Sources for COMET saved.")


In [ ]:
# =========================================================
# 13. Compute COMET Score
# =========================================================
print("13. Computing COMET score (semantic similarity)...")
# COMET expects predictions, references, and sources as lists of strings
comet_result = comet.compute(
    predictions=decoded_preds,
    references=decoded_labels,
    sources=sources_for_comet
)

In [ ]:
# =========================================================
# 13.1 Save COMET Score
# =========================================================
import json

print(f"Saving COMET score to {COMET_RESULTS_FILE}...")
with open(COMET_RESULTS_FILE, "w") as f:
    json.dump(comet_result, f, indent=4)
print("COMET score saved.")


In [ ]:
# =========================================================
# 14. Print Final Evaluation Results
# =========================================================
print("\n=========================================================")
print("               FINAL MODEL EVALUATION RESULTS            ")
print("=========================================================")
print(f"  Evaluation Loss:    {eval_results['eval_loss']:.4f}")
print(f"  BLEU Score:         {eval_results['eval_bleu']:.2f}")
print(f"  chrF Score:         {eval_results['eval_chrf']:.2f}")
print(f"  COMET Score:        {comet_result['mean_score']:.4f}")
print("=========================================================")